In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW tvw_dim_canal                                  AS
SELECT
    tip_punto                                                                  AS TIPO_PUNTO_ATENCION
    ,CASE
        WHEN LOWER(tip_punto) IN (
            'aplicacion movil',
            'app movil'
        ) THEN 'APP MOVIL'
        WHEN LOWER(tip_punto) IN (
            'portal web',
            'web'
        ) THEN 'PORTAL WEB'
        WHEN LOWER(tip_punto) IN (
            'corresponsal',
            'corresponsalia',
            'corresponsal bancario'
        ) THEN 'CORRESPONSAL BANCARIO'
        ELSE 'SIN_CLASIFICAR'
     END                                                                        AS CANAL_DIGITAL
    ,CURRENT_DATE()                                                             AS _FECHA_CARGA
    ,'silver.cleaned.tb_sucursales_red'                                         AS _FUENTE
FROM silver.cleaned.tb_sucursales_red;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.financiero.dim_canal (

    TIPO_PUNTO_ATENCION                         STRING
    ,CANAL_DIGITAL                              STRING
    ,_FECHA_CARGA                               DATE
    ,_FUENTE                                    STRING

)
USING DELTA
LOCATION 'abfss://gold@stdataknowdeveastus001.dfs.core.windows.net/financiero/dim_canal';

In [0]:
%sql
DELETE FROM gold.financiero.dim_canal;

INSERT INTO gold.financiero.dim_canal
SELECT * FROM tvw_dim_canal;